## Validation Script for Porting the Nugraph DA code 

#### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [1]:
%load_ext autoreload
%autoreload 2

#### Append the path of the Nugraph Base conda libraries

In [2]:
import os, sys
sys.path.append('/home/hnayak/.conda/envs/NugraphBase/lib/python3.10/site-packages')

#### Import packages for training

In [3]:
from pathlib import Path
import nugraph as ng
import pytorch_lightning as pl
print(ng.__file__)

/home/hnayak/nugraph_project/nugraph/nugraph/nugraph/__init__.py


#### Determine the run configuration 

In [4]:
input_files = 2 #1 or 2
trainer_epochs = 50
data_batch_size = 64

fnc_name = "dann" #dann, mmd, semantic, sinkhorn, None (if DA off)

norm_mode = "shared" # separate, shared

run_event_decoder = False
run_semantic_decoder = True
run_filter_decoder = True

run_domain_adaptation = True

model_da_loss_fnc_name = None if input_files == 1 else fnc_name
print( "model_da_loss_fnc_name [", model_da_loss_fnc_name.upper(), "]" )

if run_domain_adaptation:
   model_warmup_epochs = 1 #0 if input_files == 1 else 10
else:
   model_warmup_epochs = trainer_epochs

subname = fnc_name.upper() if run_domain_adaptation else "OFF"

if input_files == 1:
   os.environ["NUGRAPH_LOG"]='/scratch/7DayLifetime/hnayak/nugraph_data/nugraph_logs/NuGraphMain/NoDA/'
else:
   os.environ["NUGRAPH_LOG"]='/scratch/7DayLifetime/hnayak/nugraph_data/nugraph_logs/NuGraphMain/DA-%s' % fnc_name.upper()

# if input_files == 1:
#    os.environ["NUGRAPH_LOG"]='/home/%s/NuGraphLogs/NuGraphMain/NOMINAL-EPOCH%d-WARMUP%d' % (user,trainer_epochs,model_warmup_epochs)
# else:
#    os.environ["NUGRAPH_LOG"]='/home/%s/NuGraphLogs/NuGraphMain/DA-%s-EPOCH%d-WARMUP%d' % (user,subname,trainer_epochs,model_warmup_epochs)

# print( "Files are in the directory [", os.environ["NUGRAPH_LOG"], "]" )

model_da_loss_fnc_name [ DANN ]


#### Set the data and module to use

In [5]:
Data  = ng.data.NuGraphDataModule
Model = ng.models.NuGraph3

#### Declare and configure the data module

In [6]:
source_filename="/scratch/7DayLifetime/cerati/concat-final-makeup.tiny.gnn.h5"
target_filename = "/scratch/7DayLifetime/hnayak/nugraph_data/noisy_data/rms_scaled_noise/concat-final-makeup.tiny.gnn.rescaled.h5" # scaled rms wrt numiallr2
# target_filename="/scratch/7DayLifetime/hnayak/nugraph_data/noisy_data/20_percent_noise/concat-final-makeup.tiny.gnn.noisy_gaussian.h5"
# target_filename="/scratch/7DayLifetime/hnayak/nugraph_data/noisy_data/40_percent_noise/concat-final-makeup.tiny.gnn.noisy_gaussian.h5" # 40% noise
# target_filename= "/scratch/7DayLifetime/cerati/concat-final-makeup.tiny.gnn.var.h5"  ## Fixed noise

# source label dataset loss, and DA loss, remove target loss
# reapply the normalization for source to target
# for event decoder just loss source
# for semantic loss source + DA

if input_files == 1:
   nudata = Data(model=Model, batch_size=data_batch_size, data_source_path=source_filename)
elif input_files == 2:
   nudata = Data(model=Model, batch_size=data_batch_size, data_source_path=source_filename, data_target_path=target_filename)

nudata.event_classes = ['cc_nue', 'cc_numu', 'cc_nutau', 'nc']
print(nudata)
print(nudata.semantic_classes)
print(nudata.event_classes)

{Train dataloader: size=19868}
{Validation dataloader: size=1103}
{Test dataloader: size=1103}
{Predict dataloader: None}
['MIP', 'HIP', 'shower', 'michel', 'diffuse']
['cc_nue', 'cc_numu', 'cc_nutau', 'nc']


In [7]:
# dir(nudata)

In [8]:
# nudata.combined_test[0][0]['hit'].y_semantic

#### Configure network

In [9]:
nugraph = Model(
    in_features=5,  
    hit_features=128,
    nexus_features=32,
    instance_features=32,
    interaction_features=32,
    semantic_classes=nudata.semantic_classes, 
    event_classes=nudata.event_classes,
    num_iters=5,
    event_head=run_event_decoder,
    semantic_head=run_semantic_decoder,
    filter_head=run_filter_decoder,
    vertex_head=False,
    instance_head=False,
    use_checkpointing=True,
    lr=0.001,
    da_loss_fnc_name=model_da_loss_fnc_name,
    warmup_epochs=model_warmup_epochs,
    norm_mode=norm_mode)

Turning the semantic head on
turning on the temperature for target and DA
Turning the filter head on
turning on the temperature for target and DA
semantic_decoder.assume_target_label_known: False
filter_decoder.assume_target_label_known: False


#### Configure logger and callbacks
Declare a TensorBoard logger and define the output directory, so we can monitor network training. Also, define a callback so we can monitor learning rate evolution.

In [10]:
from pytorch_lightning.loggers import TensorBoardLogger
from datetime import datetime
now = datetime.now()
folder_name = "uBooNE-Tiny-Data-%s" % now.strftime("%Y-%m-%d-%H")

In [11]:
logdir = Path(os.environ["NUGRAPH_LOG"])
logdir.mkdir(parents=True, exist_ok=True)
logger = TensorBoardLogger(save_dir=logdir,name=folder_name) 
callbacks = [ 
    pl.callbacks.LearningRateMonitor(logging_interval="step"),
    pl.callbacks.ModelCheckpoint(monitor="loss/val", mode="min"),
]

#### Declare trainer and run training
First, we set the training device. To train with a GPU, pass an integer; otherwise, it defaults to CPU training. We then instantiate a PyTorch Lightning trainer and run the training stage, iterating over all batches in the training and validation datasets to optimize model parameters, logging metrics to TensorBoard.

In [12]:
device_number = 0
accelerator, devices = ng.util.configure_device(device_number)
print(accelerator)
print(devices)
# limit_train_batches=1,
# limit_val_batches=1,

trainer = pl.Trainer(accelerator=accelerator,
                     devices=devices,
                     max_epochs=trainer_epochs,
                     logger=logger,
                     callbacks=callbacks)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


gpu
[0]


In [ ]:
trainer.fit(nugraph, datamodule=nudata)
trainer.test(datamodule=nudata)

# if just inference, uncomment the following and comment  the above two lines
# 2. Define the path to your best/latest checkpoint file
# checkpoint_file = "/scratch/7DayLifetime/hnayak/nugraph_data/nugraph_logs/NuGraphMain/DA-DANN/uBooNE-Tiny-Data-2026-07-31-22/version_0/checkpoints/epoch=49-step=15500.ckpt" # <-- Update this path!
# trainer.test(model=nugraph, datamodule=nudata, ckpt_path=checkpoint_file)


You are using a CUDA device ('NVIDIA A100 80GB PCIe MIG 4g.40gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/hnayak/.conda/envs/NugraphBase/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

  | Name             | Type            | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | encoder          | Encoder         | 977    | train | 0    
1 | core_net         | NuGraphCore     | 141 K  | train | 0    
2 | semantic_decoder | Seman

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/hnayak/nugraph_project/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'y_vtx'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):



 Validation epoch ended!


/home/hnayak/.conda/envs/NugraphBase/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/hnayak/.conda/envs/NugraphBase/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

Validation: |          | 0/? [00:00<?, ?it/s]


 Validation epoch ended!


/home/hnayak/.conda/envs/NugraphBase/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/hnayak/.conda/envs/NugraphBase/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



 Finished training epoch


/home/hnayak/nugraph_project/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'y_vtx'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):


## Just inference